# Лабораторная работа: Шумоподавление

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
(a,_),(b,_)=keras.datasets.mnist.load_data()
a=a[...,None].astype('float32')/255;b=b[...,None].astype('float32')/255
rng=np.random.default_rng(42)
def noise(x):return np.clip(x+rng.normal(0,.35,x.shape),0,1).astype('float32')
an=noise(a);bn=noise(b)
inp=keras.Input((28,28,1));x=layers.Conv2D(32,3,padding='same',activation='relu')(inp)
x=layers.MaxPooling2D(2,padding='same')(x);x=layers.Conv2D(64,3,padding='same',activation='relu')(x)
x=layers.MaxPooling2D(2,padding='same')(x);x=layers.UpSampling2D(2)(x)
x=layers.Conv2D(32,3,padding='same',activation='relu')(x);x=layers.UpSampling2D(2)(x)
model=keras.Model(inp,layers.Conv2D(1,3,padding='same',activation='sigmoid')(x))
model.compile('adam','mse');history=model.fit(an,a,validation_split=.1,epochs=8,batch_size=128)
pd.DataFrame(history.history).plot(grid=True);plt.show()
p=model.predict(bn[:6])
fig,axs=plt.subplots(3,6,figsize=(12,6))
for i in range(6):
 for j,img in enumerate((b[i],bn[i],p[i])):axs[j,i].imshow(img.squeeze(),cmap='gray');axs[j,i].axis('off')
plt.show()